#  Feedback Aspect-Based Sentiment Analysis with Explainable AI of App Reviews


## Explainable AI Section


### Utilities

In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
MODEL_DIR = "/content/distilroberta-model"
PREDS_CSV = "/content/review_preds.csv"

from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).eval().to("cuda" if torch.cuda.is_available() else "cpu")

#### Stop-Word filter Customization

[The Full List of Negation and
Intensity words](https://aclanthology.org/attachments/P17-1154.Notes.pdf)

In [16]:
import spacy

nlp = spacy.load("en_core_web_sm")
stopwords = nlp.Defaults.stop_words
#print(stopwords)


# The following list is from: https://aclanthology.org/attachments/P17-1154.Notes.pdf
# Removal of negation words
stopwords -= {"no", "not", "none", "never", "neither", "nobody",
"nothing", "nowhere", "seldom", "scarcely",
"hardly", "barely", "is not", "cannot", "may not",
"could not", "would not", "did not", "do not",
"does not", "was not", "are not", "were not"}

# Removal of intensity words
stopwords -= {"awfully", "extraordinary", "unusual", "much",
"rather", "very", "entirely", "greatly", "really",
"exceedingly", "too", "completely", "terribly",
"perfectly", "quite", "certainly", "especially",
"extremely", "fairly", "highly", "increasingly",
"much more", "particularly", "probably",
"more", "absolutely", "intensely", "supremely",
"most", "pretty"}

def is_stopword(token_text: str) -> bool:
    """
    Returns True if token is a stopword (excluding key sentiment words).
    Tokenization-insensitive: works on plain text strings.
    """
    if not token_text or not isinstance(token_text, str):
        return True
    word = token_text.strip().lower()
    # also skip pure punctuation or numeric tokens
    if all(ch in ".,!?;:-–—'\"()[]{}" for ch in word):
        return True
    if word.isdigit():
        return True
    return False

In [3]:
import re, string, numpy as np
from typing import List, Tuple, Optional, Dict
import torch


def clean_span(text: str) -> str:
  """Trim punctuation and whitespace around extracted phrase"""
  return re.sub(r"^\W+|\W+$", "", text.strip())


def is_informative(token: str) -> bool:
    """Filter tokens using spaCy stopwords + punctuation."""
    t = token.lower().strip()
    if not t:
        return False
    if t in nlp.Defaults.stop_words:
        return False
    if all(ch in string.punctuation for ch in t):
        return False
    return True

def phrases_from_offsets(text: str, seq_ids: List[Optional[int]],
                         offsets: List[Tuple[int,int]],
                         scores: np.ndarray,
                         which_seq: int = 0,
                         topk: int = 6) -> List[str]:
  "Merge top-salient tokens (by scores) into readable phrases using offsets"
  idxs = [j for j,(sid,off) in enumerate(zip(seq_ids, offsets)) if sid==which_seq and off and off[1]>off[0]]
  if not idxs:
    return []
  svals = np.array([scores[j] for j in idxs])
  order = np.argsort(-svals)
  top_idxs = [idxs[i] for i in order[:max(topk,1)]]
  top_idxs.sort()

   #merge adjacent/nearby tokens
  spans, cur = [], None
  for j in top_idxs:
    start, end = offsets[j]
    if cur is None: cur = [start,end]
    elif start <= cur[1] + 1: cur[1] = max(cur[1], end)
    else: spans.append(tuple(cur)); cur = [start,end]
  if cur: spans.append(tuple(cur))

  out = []
  seen = set()
  for s,e in spans:
    frag = clean_span(text[s:e])
    if is_informative(frag):
        key = frag.lower()
        if key not in seen:
          seen.add(key); out.append(frag)
  # backfill in case everything filtered
  if not out:
    for j in top_idxs:
      s,e = offsets[j]
      frag = clean_span(text[s:e])
      if frag: out.append(frag)
      if len(out) >= topk: break
  return out[:topk]


### Integrated Gradients Captum

In [7]:
!pip install -q captum
!pip install -q tqdm

In [20]:
import os, gc
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from captum.attr import IntegratedGradients

# ---------------- CONFIG ------------------------
INPUT_CSV = PREDS_CSV
OUTPUT_CSV = "/content/review_preds_with_xai.csv"
BATCH = 4
MAX_LEN = 128
IG_STEPS = 16
TOPK = 6
USE_ASPECT_PAIR = True
ASPECT_FALLBACK = None

device = next(model.parameters()).device
model.to(device).eval()

# Loading input data
df = pd.read_csv(INPUT_CSV)
text_col = "text" if "text" in df.columns else None
aspect_col = "aspect" if "aspect" in df.columns else None
if not text_col or not aspect_col:
  raise ValueError(f"Could not find text and aspect columns in {INPUT_CSV}; columns are {df.column.tolist()}")
texts = df[text_col].astype(str).tolist()
aspects = df[aspect_col].astype(str).tolist() if (aspect_col and USE_ASPECT_PAIR) else [None]*len(texts)

# Integrated Gradients Captum Setup
emb_layer = model.get_input_embeddings()
ig = IntegratedGradients(lambda emb, attn: model(inputs_embeds = emb,
                                                 attention_mask = attn,
                                                 return_dict = True).logits)
def tokenize_batch(t_list, a_list):
  enc = tokenizer(
      t_list,
      text_pair = a_list,
      padding = True,
      truncation = True,
      max_length = MAX_LEN,
      return_tensors = "pt",
      return_offsets_mapping = True
  )
  offsets = enc.pop("offset_mapping")
  encs = enc.encodings
  return enc, offsets, encs

def ig_batch(t_list, a_list):
    enc, offsets, encs = tokenize_batch(t_list, a_list)   # returns enc, offsets, encs
    ids = enc["input_ids"].to(device)
    attn = enc["attention_mask"].to(device)

    # forward for targets
    with torch.no_grad():
        logits = model(input_ids=ids, attention_mask=attn, return_dict=True).logits
        probs  = torch.softmax(logits, dim=-1)
        preds  = probs.argmax(dim=-1)

    # embeddings & baselines
    embeds = emb_layer(ids)
    pad_id = tokenizer.pad_token_id
    if pad_id is None:
      pad_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 0
    base_ids = torch.full_like(ids, pad_id)
    base_embeds = emb_layer(base_ids)

    # IG wrt predicted class
    atts = ig.attribute(
        inputs=embeds,
        baselines=base_embeds,
        additional_forward_args=(attn,),
        target=preds,
        n_steps=IG_STEPS
    )

    token_scores = atts.sum(dim=-1).detach().cpu().numpy()
    return token_scores, probs.cpu().numpy(), preds.cpu().numpy(), offsets.cpu().tolist(), encs

def extract_top_phrases(text, seq_ids, offsets, scores, topk=TOPK):
  """
    text:   original review string
    seq_ids: list[int|None] from fast tokenizer (0=text, 1=aspect, None=special)
    offsets: list[[start,end]] per token
    scores: 1D array-like per-token attribution scores for this example
    topk:   number of phrases to return for support/oppose
    Returns: (support_phrases, oppose_phrases)
  """
  idxs = [j for j, (sid, off) in enumerate(zip(seq_ids, offsets)) if sid == 0 and off and off[1] > off[0]]
  if not idxs:
    return [], []

  s = np.asarray(scores, dtype=float)
  svals = s[idxs] if s.ndim == 1 else np.array([scores[i] for i in idxs], dtype=float)
  pos_ord = np.argsort(-svals)[:max(1, topk)]
  neg_ord = np.argsort( svals)[:max(1, topk)]
  pos_token_ids = sorted([idxs[k] for k in pos_ord])
  neg_token_ids = sorted([idxs[k] for k in neg_ord])

  def merge(token_ids):
    spans, cur = [], None
    for j in token_ids:
      s0, e0 = offsets[j]
      if cur is None: cur = [s0, e0]
      elif s0 <= cur[1]+1: cur[1] = max(cur[1], e0)
      else: spans.append(tuple(cur)); cur = [s0, e0]
    if cur: spans.append(tuple(cur))

    out, seen = [], set()
    for s0, e0 in spans:
      frag = clean_span(text[s0:e0])
      if frag and not is_stopword(frag):
        key = frag.lower()
        if key not in seen:
          seen.add(key); out.append(frag)
    return out[:topk]

  support = merge(pos_token_ids)
  oppose  = merge(neg_token_ids)
  return support, oppose

# id2label mapping
ID2LABEL = {0:"negative", 1:"neutral", 2:"positive"}

# Attribution loop
support_list, oppose_list, pred_labels, pred_probs = [], [], [], []

with tqdm(total = len(texts), desc = "Using IG Captum for Explaining", ncols = 90) as pbar:
  for i0 in range(0, len(texts), BATCH):
        b_texts, b_aspects = texts[i0:i0+BATCH], aspects[i0:i0+BATCH]
        scores, probs, preds, offs, encs = ig_batch(b_texts, b_aspects)
        for bi, text in enumerate(b_texts):
            seq_ids = encs[bi].sequence_ids
            pos, neg = extract_top_phrases(text, seq_ids, offs[bi], scores[bi], topk=TOPK)
            pred_idx = int(preds[bi])
            pred_labels.append(ID2LABEL.get(pred_idx, str(pred_idx)))
            pred_probs.append(probs[bi].tolist())
            support_list.append("; ".join(pos))
            oppose_list.append("; ".join(neg))
        pbar.update(len(b_texts))
        del scores, probs, preds, offs, encs
        torch.cuda.empty_cache(); gc.collect()

# Merge and export
out_df = df.copy()
out_df["support"] = support_list
out_df["oppose"] = oppose_list
out_df["pred_label_xai"] = pred_labels
out_df["pred_probs_xai"] = pred_probs
out_df["evidence_support_topk"] = support_list
out_df["evidence_oppose_topk"] = oppose_list
out_df["explanation_extractive"] = out_df.apply(
    lambda r: f"Predicted {r['pred_label_xai']}. Supporting cues: {r['evidence_support_topk']}. Opposing cues: {r['evidence_oppose_topk']}.",
    axis=1
)

out_df.to_csv(OUTPUT_CSV, index=False)
print(f"Explanations saved to: {OUTPUT_CSV}")

Using IG Captum for Explaining:   0%|                             | 0/972 [00:00<?, ?it/s]

Explanations saved to: /content/review_preds_with_xai.csv
